# AgroNova inference (Colab)
1. Runtime → Change runtime type → **T4 GPU**
2. **Run all** (tunggu ±5-10 menit pertama kali)
3. Salin `INFERENCE_URL` dan `INFERENCE_API_KEY` di output sel terakhir ke `apps/api/.env`

In [ ]:
REPO = "https://github.com/WANGDELL30/agronova-poultry-dashboard"
BRANCH = "feat/inference-local"   # ganti ke main setelah di-merge
!rm -rf repo && git clone -q -b $BRANCH $REPO repo
%cd repo/apps/inference
!pip -q install -r requirements.txt -r requirements-real.txt
# Colab membawa torchao lama yang bentrok dengan transformers baru; tidak dipakai di sini
!pip -q uninstall -y torchao

In [ ]:
import os, secrets, subprocess, time, urllib.request, json
os.environ["INFERENCE_MODE"] = "real"
subprocess.run("pkill -f '[u]vicorn app.main'; pkill -f '[c]f tunnel'", shell=True)  # matikan server/tunnel sisa run sebelumnya
os.environ["INFERENCE_API_KEY"] = secrets.token_urlsafe(24)  # key baru tiap run, kecuali diset di Secrets
# Hanya perlu kalau repo adapter masih private (Colab: ikon kunci -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
try:  # opsional: key tetap -> isi sekali di Colab Secrets (INFERENCE_API_KEY) dan di apps/api/.env
    os.environ["INFERENCE_API_KEY"] = userdata.get("INFERENCE_API_KEY")
except Exception:
    pass
server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
body = None
for _ in range(120):   # tunggu model siap (maks ±20 menit)
    try:
        body = json.load(urllib.request.urlopen("http://localhost:8000/health"))
        if body.get("status") == "ready": break
    except Exception:
        pass
    time.sleep(10)
print("health:", body)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cf && chmod +x cf
import re
tunnel = subprocess.Popen(["./cf", "tunnel", "--url", "http://localhost:8000"],
                          stderr=subprocess.PIPE, text=True)
url = None
while url is None:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stderr.readline())
    if m: url = m.group(0)
print("\nSalin ke apps/api/.env:\n")
print("INFERENCE_URL=" + url)
print("INFERENCE_API_KEY=" + os.environ["INFERENCE_API_KEY"])
print("INFERENCE_TIMEOUT_MS=120000")
print("\nJangan tutup tab ini selama website dipakai.")